# 02 — Extract seasonal NDVI metrics

Describe vegetation seasonality from the monthly rolling NDVI rasters generated in notebook 01. This branch supports the POS–MVC comparison and is optional for the annual productivity branch.


## Inputs, outputs, and execution order

**Inputs:** All 12 monthly rasters from notebook 01 for each requested year.

**Outputs:** Per-year phenology CSVs, seasonal-curve and spatial-panel PNGs, and a peak-magnitude GeoTIFF under `pheno/graph/`.

Run cells from top to bottom in a fresh kernel. Open this notebook beside `workflow_config.json` and `workflow_support.py`. The configuration uses Mion study settings; the original boundary and final criteria table were not supplied. See the README for the small example and for adapting the workflow.

### How to interpret this step

SOS, POS, and EOS are seasonal dates; vSOS, vPOS, and vEOS are NDVI magnitudes. LOS is a duration in days. The raster named `_POS_YYYY.tif` stores **vPOS magnitude**, while the CSV `POS` column stores the peak date. Phenology of the spatial-median curve differs from the spatial median of pixelwise phenology. Month-start labels and smoothing limit date precision; a single calendar-year cycle can be unsuitable for bimodal or cross-year seasons.


## 1. Import packages

Use the Digital Earth Africa Sandbox for steps that access the data cube. Local raster processing uses the packages listed in `environment.yml`.


In [ ]:
%matplotlib inline
import os
import glob
import re
import gc
import psutil
import datetime as dt
import numpy as np
import pandas as pd
import xarray as xr
import rioxarray as rxr
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.dates
import mpl_toolkits.axisartist as AA
from mpl_toolkits.axes_grid1 import host_subplot
from deafrica_tools.temporal import xr_phenology


## 2. Load shared settings

The settings below describe the run. Edit the JSON configuration before continuing; later parameter cells explain the settings used by this notebook.


In [ ]:
# Edit workflow_config.json once, then restart and rerun each notebook.
from workflow_support import load_config, require_files, load_aoi, assert_same_grid
cfg = load_config()
print(f"Study: {cfg['study_id']} | Years: {cfg['start_year']}-{cfg['end_year']}")
print(f"Output directory: {cfg['output_dir']}")


## 3. Set phenology inputs

The complete configured time range is used here and in notebook 03. Change `method_sos` and `method_eos` only with a documented seasonal interpretation. The existing first/last choices are retained.


In [ ]:
veg_proxy = "NDVI"
area_of_interest = cfg["study_id"]
input_dir = f"{cfg['output_dir']}/data"
output_folder = f"{cfg['output_dir']}/pheno/graph"
os.makedirs(output_folder, exist_ok=True)
basic_pheno_stats = ["SOS", "vSOS", "POS", "vPOS", "EOS", "vEOS", "Trough", "LOS", "AOS", "ROG", "ROS"]
method_sos, method_eos = "first", "last"


## 4. Define plotting and memory helpers

The seasonal-curve plot uses calendar-month labels.


In [ ]:
# =========================================================
# HELPER: RAM MONITOR
# =========================================================
def print_ram(stage=""):
    process = psutil.Process(os.getpid())
    mem_gb = process.memory_info().rss / (1024 ** 3)
    print(f"[RAM] {stage}: {mem_gb:.2f} GB")


# =========================================================
# HELPER: FORMAT X AXIS
# =========================================================
def adjust_axes(ax):
    ax.xaxis.set_major_locator(mpl.dates.MonthLocator())
    ax.xaxis.set_minor_locator(mpl.dates.MonthLocator(bymonthday=16))
    ax.xaxis.set_major_formatter(mpl.ticker.NullFormatter())
    ax.xaxis.set_minor_formatter(mpl.dates.DateFormatter("%b"))

    ax.axis["bottom"].minor_ticks.set_visible(False)
    ax.axis["top"].major_ticks.set_visible(False)
    ax.axis["top"].minor_ticks.set_visible(False)
    ax.axis["right"].major_ticks.set_visible(False)
    ax.axis["right"].minor_ticks.set_visible(False)


## 5. Calculate yearly seasonal metrics

Each year is loaded into RAM, summarized spatially, and processed per pixel. Date fields and numeric duration fields are written separately. Missing input months or failed years stop the notebook before a combined table is produced.


In [ ]:
# =========================================================
# MAIN LOOP
# =========================================================
failed_years = []
for target_year in range(cfg["start_year"], cfg["end_year"] + 1):
    print("\n" + "=" * 70)
    print(f"Processing year: {target_year}")
    print("=" * 70)
    print_ram("Start of year")

    # Predefine variables so cleanup is safe
    datasets = []
    times = []
    ds = None
    veg_smooth_1D = None
    grouped_1d = None
    pheno_results_1d = None
    df_dict = None
    df = None
    sub = None
    grouped_spatial = None
    pheno_results_spatial = None
    phen = None
    fig = None
    ax = None
    da = None
    d = None

    try:
        # =========================================================
        # SETTINGS
        # =========================================================
        pattern = f"ndvi_roll3_{target_year}??_{cfg['file_version']}.tif"

        # =========================================================
        # FIND TIFF FILES
        # =========================================================
        tif_files = sorted(glob.glob(os.path.join(input_dir, pattern)))

        if len(tif_files) == 0:
            raise FileNotFoundError(
                f"No TIFF files found for year {target_year} in: {input_dir}"
            )

        if len(tif_files) != 12:
            raise ValueError(f"Expected 12 monthly files for {target_year}, found {len(tif_files)}")
        assert_same_grid(tif_files)
        print(f"Found {len(tif_files)} TIFF files for {target_year}")

        # =========================================================
        # BUILD XARRAY TIME SERIES
        # =========================================================
        for f in tif_files:
            match = re.search(r"(\d{6})", os.path.basename(f))
            if not match:
                raise ValueError(f"Date not found in filename: {f}")

            yyyymm = match.group(1)
            timestamp = pd.to_datetime(yyyymm, format="%Y%m")

            # Open, load into memory, then close file handle immediately
            da = rxr.open_rasterio(f, masked=True).squeeze("band", drop=True)
            da = da.load()

            try:
                da.close()
            except Exception:
                pass

            datasets.append(da)
            times.append(timestamp)

            # Clear loop variable reference
            da = None

        ds = xr.concat(datasets, dim="time")
        ds = ds.assign_coords(time=("time", times))

        print(ds)
        print_ram("After building xarray time series")
        print("\n✓ Full xarray time series created successfully.")

        # Free source lists as soon as concat is done
        for d in datasets:
            try:
                d.close()
            except Exception:
                pass
        datasets.clear()
        times.clear()
        datasets = None
        times = None
        d = None
        gc.collect()
        print_ram("After concat cleanup")

        # =========================================================
        # MEDIAN TIME SERIES FOR WHOLE AOI
        # =========================================================
        veg_smooth_1D = ds.median(dim=["x", "y"], skipna=True)
        print_ram("After AOI median time series")

        # =========================================================
        # CALCULATE 1D PHENOLOGY
        # =========================================================
        grouped_1d = veg_smooth_1D.groupby("time.year")
        pheno_results_1d = {}

        for yr, da_year in grouped_1d:
            stats = xr_phenology(
                da_year,
                method_sos=method_sos,
                method_eos=method_eos,
                stats=basic_pheno_stats,
                verbose=False,
            )
            pheno_results_1d[str(yr)] = stats

        print_ram("After 1D phenology")

        # =========================================================
        # CONVERT 1D PHENOLOGY TO DATAFRAME
        # =========================================================
        df_dict = {}

        for key, value in pheno_results_1d.items():
            row_dict = {}

            for var_name in value.data_vars:
                arr = value[var_name]

                # Only date fields become timestamps; LOS and NDVI metrics stay numeric.
                if var_name in {"SOS", "POS", "EOS"}:
                    result = pd.to_datetime(arr.values)
                elif np.issubdtype(arr.dtype, np.timedelta64):
                    result = float(arr.values / np.timedelta64(1, "D"))
                else:
                    result = round(float(arr.values), 3)

                row_dict[var_name] = result

            df_dict[key] = row_dict

        df = pd.DataFrame(df_dict).T

        csv_path = os.path.join(
            output_folder,
            f"{area_of_interest}_csv_{veg_proxy}_{target_year}.csv"
        )
        df.to_csv(csv_path)
        print(f"Saved: {csv_path}")

        # =========================================================
        # PLOT 1D YEARLY CURVE
        # =========================================================
        sub = veg_smooth_1D.sel(time=veg_smooth_1D["time.year"] == target_year)

        if sub.time.size == 0:
            raise ValueError(f"No time series values found for {target_year}")

        fig = plt.figure(figsize=(15, 7))
        ax = host_subplot(111, figure=fig, axes_class=AA.Axes)

        sub.plot(ax=ax, label=str(target_year))

        yr_str = str(target_year)

        sos = df.loc[yr_str, "SOS"]
        eos = df.loc[yr_str, "EOS"]
        pos = df.loc[yr_str, "POS"]

        vsos = df.loc[yr_str, "vSOS"]
        veos = df.loc[yr_str, "vEOS"]
        vpos = df.loc[yr_str, "vPOS"]

        ax.plot(sos, vsos, "or")
        ax.annotate(
            "SOS",
            xy=(sos, vsos),
            xytext=(-15, 20),
            textcoords="offset points",
            arrowprops=dict(arrowstyle="-|>")
        )

        ax.plot(eos, veos, "or")
        ax.annotate(
            "EOS",
            xy=(eos, veos),
            xytext=(0, 20),
            textcoords="offset points",
            arrowprops=dict(arrowstyle="-|>")
        )

        ax.plot(pos, vpos, "or")
        ax.annotate(
            "POS",
            xy=(pos, vpos),
            xytext=(-10, -25),
            textcoords="offset points",
            arrowprops=dict(arrowstyle="-|>")
        )

        ax.set_xlim(dt.date(target_year, 1, 1), dt.date(target_year, 12, 31))

        ymin = float(sub.min(skipna=True).values)
        ymax = float(sub.max(skipna=True).values)
        ax.set_ylim([ymin - 0.025, ymax + 0.05])

        adjust_axes(ax)
        ax.legend(labelcolor="linecolor")
        plt.ylabel(veg_proxy)
        plt.xlabel("Month")
        plt.title(f"Yearly {veg_proxy} — {target_year}")
        plt.tight_layout()

        chart_path = os.path.join(
            output_folder,
            f"{area_of_interest}_{veg_proxy}_chart_{target_year}.png"
        )
        plt.savefig(chart_path, dpi=400, bbox_inches="tight")
        print(f"Saved: {chart_path}")
        plt.close(fig)
        fig = None
        ax = None

        print_ram("After yearly curve plot")

        # =========================================================
        # CALCULATE SPATIAL PHENOLOGY
        # =========================================================
        grouped_spatial = ds.groupby("time.year")
        pheno_results_spatial = {}

        for yr, da_year in grouped_spatial:
            stats = xr_phenology(
                da_year,
                method_sos=method_sos,
                method_eos=method_eos,
                stats=basic_pheno_stats,
                verbose=False,
            )
            pheno_results_spatial[str(yr)] = stats

        phen = pheno_results_spatial[str(target_year)]
        print_ram("After spatial phenology")

        # =========================================================
        # SPATIAL PHENOLOGY PANEL
        # =========================================================
        fig, ax = plt.subplots(
            nrows=2,
            ncols=5,
            figsize=(18, 8),
            sharex=True,
            sharey=True
        )

        cbar_size = 0.7

        phen.SOS.dt.dayofyear.plot(
            ax=ax[0, 0],
            cmap="magma_r",
            add_colorbar=True,
            cbar_kwargs=dict(shrink=cbar_size, label=None),
        )
        ax[0, 0].set_title("Start of Season")

        phen.vSOS.plot(
            ax=ax[0, 1],
            cmap="YlGn",
            vmax=0.8,
            cbar_kwargs=dict(shrink=cbar_size, label=None),
        )
        ax[0, 1].set_title(f"{veg_proxy} at SOS")

        phen.POS.dt.dayofyear.plot(
            ax=ax[0, 2],
            cmap="magma_r",
            add_colorbar=True,
            cbar_kwargs=dict(shrink=cbar_size, label=None),
        )
        ax[0, 2].set_title("Peak of Season")

        phen.vPOS.plot(
            ax=ax[0, 3],
            cmap="YlGn",
            vmax=1,
            cbar_kwargs=dict(shrink=cbar_size, label=None),
        )
        ax[0, 3].set_title(f"{veg_proxy} at POS")

        phen.EOS.dt.dayofyear.plot(
            ax=ax[0, 4],
            cmap="magma_r",
            add_colorbar=True,
            cbar_kwargs=dict(shrink=cbar_size, label=None),
        )
        ax[0, 4].set_title("End of Season")

        phen.vEOS.plot(
            ax=ax[1, 0],
            cmap="YlGn",
            vmax=0.8,
            cbar_kwargs=dict(shrink=cbar_size, label=None),
        )
        ax[1, 0].set_title(f"{veg_proxy} at EOS")

        phen.LOS.plot(
            ax=ax[1, 1],
            cmap="magma_r",
            vmax=300,
            vmin=0,
            cbar_kwargs=dict(shrink=cbar_size, label=None),
        )
        ax[1, 1].set_title("Length of Season (Days)")

        phen.AOS.plot(
            ax=ax[1, 2],
            cmap="YlGn",
            vmax=0.8,
            cbar_kwargs=dict(shrink=cbar_size, label=None),
        )
        ax[1, 2].set_title("Amplitude of Season")

        phen.ROG.plot(
            ax=ax[1, 3],
            cmap="coolwarm_r",
            vmin=-0.02,
            vmax=0.02,
            cbar_kwargs=dict(shrink=cbar_size, label=None),
        )
        ax[1, 3].set_title("Rate of Growth")

        phen.ROS.plot(
            ax=ax[1, 4],
            cmap="coolwarm_r",
            vmin=-0.02,
            vmax=0.02,
            cbar_kwargs=dict(shrink=cbar_size, label=None),
        )
        ax[1, 4].set_title("Rate of Senescence")

        plt.suptitle(f"Per Pixel Phenology for {target_year}")
        plt.tight_layout()

        spatial_png = os.path.join(
            output_folder,
            f"{area_of_interest}_{veg_proxy}_spatial_{target_year}.png"
        )
        plt.savefig(spatial_png, dpi=400, bbox_inches="tight")
        print(f"Saved: {spatial_png}")
        plt.close(fig)
        fig = None
        ax = None

        # =========================================================
        # SAVE vPOS TIFF
        # =========================================================
        vpos_tif = os.path.join(
            output_folder,
            f"{area_of_interest}_{veg_proxy}_POS_{target_year}.tif"
        )
        # Historical POS filename contains peak NDVI magnitude, not a date raster.
        phen.vPOS.rio.write_crs(ds.rio.crs).rio.write_nodata(np.nan).rio.to_raster(vpos_tif)
        print(f"Saved: {vpos_tif}")

        print(f"\n✓ Finished processing {target_year}")

    except Exception as e:
        failed_years.append(target_year)
        print(f"Error processing year {target_year}: {e}")

    finally:
        # =========================================================
        # MEMORY CLEANUP
        # =========================================================
        print_ram("Before cleanup")

        # Close all figures
        plt.close("all")

        # Close DataArrays still hanging around
        if datasets is not None:
            for d in datasets:
                try:
                    d.close()
                except Exception:
                    pass

        # Explicitly delete references
        datasets = None
        times = None
        ds = None
        veg_smooth_1D = None
        grouped_1d = None
        pheno_results_1d = None
        df_dict = None
        df = None
        sub = None
        grouped_spatial = None
        pheno_results_spatial = None
        phen = None
        fig = None
        ax = None
        da = None
        d = None

        gc.collect()

        print_ram("After cleanup")
        print("✓ Memory cleared for next loop")

if failed_years:
    raise RuntimeError(f"Phenology failed for years: {failed_years}; do not continue to MVC yet.")


## 6. Combine yearly summary tables

Only the configured years are merged. The combined CSV is excluded from its own inputs, making reruns safe.


In [ ]:
csv_files = [os.path.join(output_folder, f"{area_of_interest}_csv_{veg_proxy}_{y}.csv")
             for y in range(cfg["start_year"], cfg["end_year"] + 1)]
require_files(*csv_files)
df_all = pd.concat((pd.read_csv(f, index_col=0).rename_axis("Year").reset_index() for f in csv_files), ignore_index=True)
out_file = os.path.join(output_folder, f"{area_of_interest}_csv_{veg_proxy}_all_years.csv")
df_all.to_csv(out_file, index=False)
print(f"Merged {len(csv_files)} annual tables: {out_file}")


## Check the outputs before continuing

- Review warnings and errors, and confirm that every requested year or reporting period was written.
- Open an output raster and inspect its CRS, extent, nodata mask, and value range.
- Keep the configuration and software versions with your results. Run `python scripts/capture_environment.py` from the repository root.
- If you change the AOI, dates, metric, or thresholds, use a fresh output directory to avoid reusing older results.

See [README.md](README.md), [data requirements](docs/DATA_REQUIREMENTS.md), and [method details](docs/METHODS.md).

The supplied notebooks identify their code as Apache License 2.0; retain the original Digital Earth Africa attribution when reusing it. Dataset licences and acknowledgements apply separately. Use the Issues tab of the repository hosting this workflow for workflow-specific questions.

**Validation status:** documentation, syntax, and synthetic-data checks are described in `docs/VALIDATION.md`. This revised workflow has not been executed against the live Digital Earth Africa data cube. A displayed current date or package version is not evidence of a successful full run.
